In [ ]:
import pprint
import requests
import json

In [ ]:
main_url = 'https://en.wikipedia.org/w/api.php'
attr = ['Causes', 'Diagnosis', 'Medication', 
                    'Characteristics', 'Symptoms', 'Treatment']
expanded_attr = [['Causes'], ['Diagnosis'], ['Treatment', 'Medication'], 
                    ['Symptoms', 'Characteristics']]
attributes = frozenset(attr)
not_disorders = frozenset(['American', 'Board', 'psychiatric', 'Association', 'psychiatry', 'medicine', 'alternative', 'not', 'Campaign', 'against', 'withdrawl', 'intoxication'
                          'induced', 'Cathinone', 'organization', 'list', 'policy', 'intoxication'])

def query_wikipedia(**kwargs):
    req = ''
    for kw in kwargs.keys():
        req += '&' if len(req) else '?'
        k = str(kw).replace(' ', '%20')
        v = str(kwargs[kw]).replace(' ', '%20')
        req += k + '=' + v
    req += '&formatversion=latest&format=json'
    # print(url+req)
    r = requests.get(url = main_url + req)
    return r.json()

def get_disorders_from_wikipedia():
    r = query_wikipedia(action='query', prop='links',
                        list='',titles='List of mental disorders', pllimit='max')
    seen_disorders = set()
    for link in r['query']['pages'][0]['links']:
        is_not_disorder = any(e.lower() in link['title'].lower() for e in not_disorders)
        if not is_not_disorder:
            seen_disorders.add(link['title'])
    return list(seen_disorders)

def get_disorder_information(title):
    if not title:
        pass
    disorder_url = 'https://en.wikipedia.org/w/api.php?action=query&format=json&prop=extracts%7Cimages%7Cpageimages&piprop=original%7Cthumbnail&formatversion=2&explaintext=1&exsectionformat=wiki&'
    title = title.replace(' ', '%20')
    j = requests.get(url = disorder_url + '&titles=' + title).json()['query']['pages'][0]
    if 'original' in j.keys():
        image_url = j['original']['source']
    elif 'thumbnail' in j.keys():
        image_url = j['thumbnail']['source']
    else:
        image_url = ''

    return (j['extract'], image_url)

def dissect_wikipedia_article(tup):
    article, image_url = tup
    lines = article.split('\n')
    to_ret : dict = {'Description':'', 'Image url': image_url}
    i = 0
    last = 'Description'
    while i < len(lines):
        line = lines[i]
        # print(line[:10])
        if len(line) <= 3:
            # print('passing')
            pass
        elif not line[0] == '=' and last:
            # print(f'adding to {last}')
            to_ret[last] += line
        else:
            kw = [False] * len(expanded_attr)
            for n, k in enumerate(expanded_attr):
                for j in k:
                    if j.lower() in line.lower():
                        kw[n] = True
            if any(kw):
                j = kw.index(True)
                last = expanded_attr[j][0]
                to_ret[last] = ''
        i += 1
    return to_ret

def write_all_disorder_info():
    disorders = get_disorders_from_wikipedia()
    full_model = {}
    required_attr = frozenset([j[0] for j in expanded_attr] + ['Description', 'Image url'])
    missing_count = {}
    for a in required_attr:
        missing_count[a] = 0
    missing_none_count = 0
    with open('../data/Disorders.jsonl', 'w+') as f:
        for disorder in disorders:
            temp = dissect_wikipedia_article(get_disorder_information(str(disorder)))
            # temp = {disorder: }
            missing = False
            for k in missing_count.keys():
                if not (k.lower() in [b.lower() for b in temp.keys()]):
                    missing_count[k] += 1
                    missing = True
            if len(temp['Image url']) < 5:
                # missing = True
                missing_count['Image url'] += 1
            if not missing:
                missing_none_count += 1
                json.dump({disorder: temp}, f)
                f.write('\n')
    print(missing_count)
    print(len(disorders))
    print(missing_none_count)

In [ ]:
write_all_disorder_info()

In [ ]:
t1 = get_disorder_information('Pica (disorder)')
t2 = dissect_wikipedia_article(t1)
pprint.pprint(t1)
# print(t1)

In [ ]:
pprint.pprint(t2)

In [ ]:
token_endpoint = 'https://icdaccessmanagement.who.int/connect/token'
client_id = '...'
client_secret = '...'
scope = 'icdapi_access'
grant_type = 'client_credentials'

# get the OAUTH2 token

# set data to post
payload = {'client_id': client_id, 
	   	   'client_secret': client_secret, 
           'scope': scope, 
           'grant_type': grant_type}
           
# make request
r = requests.post(token_endpoint, data=payload, verify=False).json()
token = r['access_token']
print(token)


# access ICD API

uri = 'https://id.who.int/icd/entity'

# HTTP header fields to set
headers = {'Authorization':  'Bearer '+token, 
           'Accept': 'application/json', 
           'Accept-Language': 'en',
	   'API-Version': 'v2'}
           
# make request           
r = requests.get(uri, headers=headers, verify=True)

# print the result
pprint.pprint(r.json())	